# Pipeline de Análise de Sentimento e Dores do Cliente (NLP)

**Projeto:** Tech Challenge - Fase 5 (Pós-Tech Data Science)

**Tema:** Classificação de Sentimento e Topic Modeling em Reclamações Financeiras

**Arquitetura:** [Knowledge Distillation](https://www.ibm.com/think/topics/knowledge-distillation) (Teacher-Student) & [Medallion Data Architecture](https://learn.microsoft.com/en-us/azure/databricks/lakehouse/medallion)


## Visão Geral e Objetivo

O objetivo deste projeto foi desenvolver uma solução escalável de Processamento de Linguagem Natural (NLP) capaz de ingerir, classificar e analisar um grande volume de reclamações financeiras (dataset *Consumer Complaints*).

O desafio central residia na natureza dos dados: **não rotulados (unlabeled)** e volumosos (> 2GB). A solução exigia não apenas a classificação de sentimento (Positivo/Negativo), mas também a descoberta automática dos principais motivos de insatisfação ("Dores do Cliente") e a entrega de um modelo performático para inferência em produção.

### Fluxo de Dados (Pipeline)

O pipeline foi desenhado para garantir eficiência de memória (*Out-of-Core Processing*):

`RAW (Bronze)` $\to$ `LEAN (Silver)` $\to$ `LABELED (Teacher)` $\to$ `GOLD DATASET` $\to$ `STUDENT MODEL` $\to$ `INSIGHTS`

### Fase 1: Ingestão e Engenharia de Dados (Bronze $\to$ Silver)

**O Desafio:** O arquivo original (`complaints.csv.zip`) contém milhões de linhas e colunas desnecessárias, inviabilizando o carregamento total em memória (RAM) com bibliotecas tradicionais como Pandas.

**A Solução:**

- **Ferramenta:** **DuckDB** + **Polars**.

- **Técnica:** Extração do arquivo ZIP e conversão para formato parquet.

- **Ação:** Seleção apenas das colunas vitais (ID, Texto, Produto, Data) e filtragem temporal (Ano 2025).

- **Justificativa:** O DuckDB permite executar SQL em arquivos economizando armazenamento e I/O. O formato de saída **Parquet (Snappy)** reduziu o volume de dados em ~70% mantendo a performance de leitura.

### 1 - [Notebook com ingestão de dados](./1_data_ingestion.ipynb)

### Após a execução do Notebook acima 👆 temos:
Arquivo **complaints_lean_2025.parquet** em ./data/parquet/
- Total de registros: 1.192.432
- Período de 01/01/2025 a 31/12/2025
- nome das colunas em snake case


**Entrada:** aquivo zip com as reclamações  
**Saída:** arquivo parquet filtrado por ano

`RECLAMAÇÕES - RAW | complaints.csv.zip` $\to$ ` ARQUIVO Parquet | complaints_lean_2025.parquet` 

----------

### Fase 2: Criação da Variável Target (O Modelo Teacher)

**O Desafio:** Como treinar um classificador sem ter rótulos (labels) prévios?

**A Solução:**

- **Modelo:** `cardiffnlp/twitter-roberta-base-sentiment-latest`.

- **Implementação:** Inferência em *batches* na GPU com salvamento incremental (*Checkpointing*).

- **Justificativa:** Modelos baseados em Transformers (como RoBERTa) entendem contexto bidirecional. Diferente de abordagens baseadas em palavras-chave, o RoBERTa compreende que *"My issue was not resolved"* é negativo, mesmo contendo a palavra *"resolved"*.

- **Mecanismo de Segurança:** Implementação de scripts de *Resume*, permitindo que o processamento fosse interrompido e retomado sem perda de dados, crucial para processamentos longos (> 10 horas).

### 2 - [Notebook de criação da variável Target](./2_variavel_target.ipynb)

**Entrada:**  arquivo parquet filtrado por ano

**Saída:** arquivos parquet incrementais com os rótulos (data/parquet/labeled/)

`complaints.csv.zip` $\to$ `Arquivos com rótulos (data/parquet/labeled/)` 

-------------

### Fase 3: Curadoria do Dataset (Camada Gold)
**O Desafio**: O modelo Teacher não é infalível. Treinar o modelo final com dados onde o Teacher estava "em dúvida" geraria um modelo confuso.

**A Solução** (The Gold Rule):

Criação de um training_dataset.parquet consolidado.

**Filtro de Confiança**: Apenas registros onde o Teacher apresentou Score > 0.75 foram mantidos.

**Justificativa**: Eliminar o ruído. Ao treinar o Student apenas com exemplos de "alta certeza", garantimos limites de decisão mais claros e reduzimos alucinações.

### 3 - [Notebook de Curadoria do Dataset](./3_curadoria.ipynb)



**Entrada:**  arquivos parquet incrementais com os rótulos (data/parquet/labeled/) 

**Saída:** arquivos parquet único com reclamação e a váriável Target *training_dataset_2025.parquet*

`Arquivos com rótulos (data/parquet/labeled/)` $\to$ `training_dataset_2025.parquet` 

__________

### Fase 4: Pré-processamento e Compliance

**O Desafio:** Atender aos requisitos de limpeza de texto clássica de NLP sem destruir a semântica necessária para análise de sentimento.

**A Solução:**

- **Ferramenta:** **spaCy** (Industrial-strength NLP).
  
- **Lógica:**
  
  - Remoção de pontuação e caracteres especiais.
    
  - Lematização (reduzir palavras à raiz).
    
  - **Smart Stopwords:** Remoção de stopwords padrão, **EXCETO** negações (`no`, `not`, `never`, `nor`).
    
- **Justificativa:** Em análise de sentimento, a negação inverte a polaridade. Remover o "not" de "not good" alteraria o sentido para "good". Esta abordagem preservou a integridade semântica.

### 4 - [Notebook de Pré-processamento](./4_pre_processing.ipynb)

**Entrada:**  arquivo parquet único com reclamação e a váriável Target *training_dataset_2025.parquet*

**Saída:** arquivo com texto pré-processado *processed_dataset.parquet*

`training_dataset_2025.parquet` $\to$ `processed_dataset.parquet` 

--------------------

### Fase 5: Treinamento do Modelo Student (DistilBERT)

**O Desafio:** Criar um modelo leve para produção.

**A Solução:**

- **Arquitetura:** Fine-Tuning do `distilbert-base-uncased`.
  
- **Setup:** Divisão Treino/Teste (80/20) estratificada.
  
- **Métrica:** F1-Score Macro (para balancear a performance entre classes positivas e negativas).
  
- **Justificativa:** O DistilBERT retém cerca de 97% da performance do BERT/RoBERTa, sendo **40% menor e 60% mais rápido**. Ideal para ambientes produtivos onde latência é KPI.

### 5 - [Notebook de Treinamento](./5_training.ipynb)

**Entrada:**  arquivo com texto pré-processado *processed_dataset.parquet*

**Saída:** modelo treinado

`processed_dataset.parquet` $\to$ `student_model` 

_______________________

### Fase 6: Análise de Dores (Topic Modeling)

**O Desafio:** Responder à pergunta de negócio: *"Por que os clientes estão reclamando?"*

**A Solução:**

- **Algoritmo:** **BERTopic** (Embeddings + HDBSCAN + c-TF-IDF).
  
- **Escopo:** Aplicado apenas às reclamações classificadas como **Negativas**.
  
- **Resultado:** Agrupamento semântico automático (ex: cluster de "cobrança indevida", cluster de "bloqueio de conta").
  
- **Justificativa:** O BERTopic supera o LDA (Latent Dirichlet Allocation) clássico por não depender de "saco de palavras", conseguindo capturar tópicos baseados no contexto semântico das frases.

### 6 - [Notebook de Análise](./6_topic_modeling.ipynb)

**Entrada:**  arquivos *processed_dataset.parquet* e *complaints_lean_2025.parquet*

**Saída:** Pasta *reports* com as análises

`processed_dataset.parquet e complaints_lean_2025.parquet` $\to$ `reports` 

----------------------------------

### Fase 7: Comparação de Perfomance

**O Desafio:** Comparação entre o modelo completo Teacher e o modelo menor Student




### 7 - [Notebook de Comparação](./7_performance.ipynb)

**Entrada:**  modelos

**Saída:** imagem com comparação na pasta *reports/figures* 

`modelos` $\to$ `reports\figures` 